In [0]:
class Silver_drivers_pit_stops():
    main_path="/Volumes/formula1_race/default/formula1/"
    bronze_path = "formula1_race_project/bronze"
    silver_path = "formula1_race_project/silver"

    def __init__(self,table,drivers_df,races_df):#
        self.table=table
        self.drivers_df=drivers_df # intializing the broadcasted dataframe
        self.races_df=races_df # intializing the broadcasted dataframe
        
    def max_watermark_value(self):
        from pyspark.sql.functions import max,col

        #fetching max_ingestion_date from gold layer table drivers_pit_stops and printing that water mark value
        if spark.catalog.tableExists("formula1_race.silver.drivers_pit_stops"):
            pit_stops_max_ingestion_date =spark.read.table('formula1_race.silver.drivers_pit_stops').agg(max(col('pit_stops_ingestion_date')).alias('max_ingestion_date')).collect()[0]['max_ingestion_date']
            if pit_stops_max_ingestion_date is None:
                pit_stops_max_ingestion_date='1900-01-01 00:00:00'
        else:
            pit_stops_max_ingestion_date='1900-01-01 00:00:00'
        print(f"results_max_ingestion_date:{pit_stops_max_ingestion_date}")

        return pit_stops_max_ingestion_date

    def read_input(self,list_max_ingest):
        from pyspark.sql.functions import max,col,expr,count
        #fetching incremental pit_stops data
        pit_stops_max_ingestion_date=list_max_ingest
        incr_pit_stops_df= (spark.read.table('formula1_race.silver.pit_stops')
                        .filter(col('pit_stops_ingestion_date')>pit_stops_max_ingestion_date)
                            )
                          
        print("incr_pit_stops_df")
        display(incr_pit_stops_df.select(count("*")))
        # races_df=spark.read.table('formula1_race.silver.races')
        # drivers_df=spark.read.table('formula1_race.silver.drivers')

        # passing all requiried tables for join as list
        read_df_list=[self.races_df,incr_pit_stops_df,self.drivers_df]
        return read_df_list

    
    def apply_transformations(self,read_df_list):
        from pyspark.sql.functions import round,col,dense_rank,broadcast
        from pyspark.sql.window import Window
        races_df=read_df_list[0]
        pit_stops_df=read_df_list[1]
        drivers_df=read_df_list[2]
        #performing join operation only on  incremental pit_stops data with all other tables
        drivers_pit_stops_df= (pit_stops_df.join(races_df,['race_id'],'inner')
                                   .join(drivers_df,['driver_id'],'inner')
                                   .selectExpr("race_year","race_name","race_date","race_time","round as race_round","driver_name","driver_nationality","stop","lap","time","duration_sec","duration_minutes","milliseconds","pit_stops_ingestion_date")
                             )
        print("detailed execution plain")
        drivers_pit_stops_df.explain(True)
        return drivers_pit_stops_df

        
    def write_output(self,apply_tran_df):
        # writing those data into gold layer table by partitioning according to filter approach using append mode
        (apply_tran_df.write.partitionBy("race_year","race_name")
         .mode("append")
         .saveAsTable(f"formula1_race.silver.{self.table}"))
        display(spark.sql(f"select count(*) from formula1_race.silver.{self.table}"))
        print("Data write into silver drivers_pit_stops table is Done")
        
    
       
        
    def process(self):
        print("Started silver-ingestion-drivers_pit_stops  in ran....")
        list_max_ingest=self.max_watermark_value() # return last water mark value
        read_df_list=self.read_input(list_max_ingest) # return the lsit of tables
        apply_tran_df=self.apply_transformations(read_df_list) # return the joined data
        self.write_output(apply_tran_df)#write data into gold table


In [0]:
# Silver_drivers_pit_stops_instance = Silver_drivers_pit_stops("drivers_pit_stops")
# Silver_drivers_pit_stops_instance .process()
# print("Successfully Gold_drivers_pit_stops is ran")